# Light Attenuation (KD490) — All Norway

Downloads KD490 (diffuse attenuation at 490 nm) from CMEMS,
covering all of Norway using two products:
- **Atlantic** (1 km): `cmems_obs-oc_atl_bgc-transp_my_l3-multi-1km_P1D`
- **Arctic** (4 km): `cmems_obs-oc_arc_bgc-transp_my_l3-multi-4km_P1D`

Computes a multi-year seasonal mean (April–October), reprojects to
EPSG:25833, fills gaps, and classifies the photic zone at 50 m resolution.


In [15]:
from pathlib import Path

import copernicusmarine as cm
import numpy as np
from osgeo import gdal
from rasterio.warp import transform_bounds
import rasterio as rio
import geopandas as gpd

import mnk

gdal.UseExceptions()

out_dir = Path(".")

kd_start_year: int = 2020  #: First year of averaging period
kd_end_year: int = 2025  #: Last year of averaging period
kd_month_start: int = 4  #: Start month of growing season (April)
kd_month_end: int = 10  #: End month of growing season (October)


## 1. Bounding box from Norway DEM

In [16]:
dem_url = "/vsicurl/https://storage.googleapis.com/niva-geodata/MarintNaturKart/features/feature_norge_dem50_depth_filled.tif"

ds = gdal.Open(dem_url)
gt = ds.GetGeoTransform()
dem_xsize, dem_ysize = ds.RasterXSize, ds.RasterYSize
dem_left = gt[0]
dem_top = gt[3]
dem_right = dem_left + gt[1] * dem_xsize
dem_bottom = dem_top + gt[5] * dem_ysize
dem_srs = ds.GetSpatialRef()
ds = None

lon_min, lat_min, lon_max, lat_max = transform_bounds(
    "EPSG:25833", "EPSG:4326", dem_left, dem_bottom, dem_right, dem_top,
)
lon_min, lat_min = round(lon_min - 0.5, 1), round(lat_min - 0.5, 1)
lon_max, lat_max = round(lon_max + 0.5, 1), round(lat_max + 0.5, 1)

print(f"Norway bbox (WGS84): lon [{lon_min}, {lon_max}], lat [{lat_min}, {lat_max}]")
print(f"DEM grid: {dem_xsize} x {dem_ysize} @ 50 m, bounds: [{dem_left}, {dem_bottom}, {dem_right}, {dem_top}]")

Norway bbox (WGS84): lon [-2.2, 32.8], lat [57.0, 72.3]
DEM grid: 24431 x 30735 @ 50 m, bounds: [-99600.0, 6426000.0, 1121950.0, 7962750.0]


## 2. Download KD490 from CMEMS


In [17]:
start_date = f"{kd_start_year}-{kd_month_start:02d}-01"
end_date = f"{kd_end_year}-{kd_month_end:02d}-31"
print(f"Time range: {start_date} to {end_date}")
print(f"Seasonal filter: months {kd_month_start}–{kd_month_end}")

OVERLAP_LAT = 62.0

ds_atl = cm.open_dataset(
    dataset_id="cmems_obs-oc_atl_bgc-transp_my_l3-multi-1km_P1D",
    variables=["KD490"],
    minimum_longitude=lon_min,
    maximum_longitude=lon_max,
    minimum_latitude=lat_min,
    maximum_latitude=min(lat_max, 66.0),
    start_datetime=start_date,
    end_datetime=end_date,
)
print(f"Atlantic shape: {ds_atl["KD490"].shape}")


Time range: 2020-04-01 to 2025-10-31
Seasonal filter: months 4–10


INFO - 2026-08-21T20:35:10Z - Selected dataset version: "202603"
INFO - 2026-08-21T20:35:10Z - Selected dataset part: "default"
WARNING - 2026-08-21T20:35:10Z - Some of your subset selection [57.0, 66.0] for the latitude dimension exceed the dataset coordinates [20.005207061767578, 65.99478912353516]
WARNING - 2026-08-21T20:35:10Z - Some of your subset selection [-2.2, 32.8] for the longitude dimension exceed the dataset coordinates [-45.99479293823242, 12.994793891906738]
WARNING - 2026-08-21T20:35:10Z - Some of your subset selection [57.0, 66.0] for the latitude dimension exceed the dataset coordinates [20.005207061767578, 65.99478912353516]
WARNING - 2026-08-21T20:35:10Z - Some of your subset selection [-2.2, 32.8] for the longitude dimension exceed the dataset coordinates [-45.99479293823242, 12.994793891906738]


Atlantic shape: (2040, 864, 1459)


In [18]:
ds_arc = cm.open_dataset(
    dataset_id="cmems_obs-oc_arc_bgc-transp_my_l3-multi-4km_P1D",
    variables=["KD490"],
    minimum_longitude=lon_min,
    maximum_longitude=lon_max,
    minimum_latitude=OVERLAP_LAT,
    maximum_latitude=lat_max,
    start_datetime=start_date,
    end_datetime=end_date,
)
print(f"Arctic shape: {ds_arc["KD490"].shape}")


INFO - 2026-08-21T20:35:12Z - Selected dataset version: "202311"
INFO - 2026-08-21T20:35:12Z - Selected dataset part: "default"
WARNING - 2026-08-21T20:35:12Z - Some of your subset selection [62.0, 72.3] for the latitude dimension exceed the dataset coordinates [66.0, 90.00000000000091]
WARNING - 2026-08-21T20:35:12Z - Some of your subset selection [62.0, 72.3] for the latitude dimension exceed the dataset coordinates [66.0, 90.00000000000091]


Arctic shape: (2040, 210, 389)


## 3. Compute multi-year seasonal mean and mosaic

Filter to growing season months, compute per-pixel temporal mean.
Prefer Atlantic (1 km) where available; fill gaps with Arctic (4 km, linearly interpolated).


In [19]:
kd_merged, lons, lats = mnk.light.merge_kd490_datasets(
    ds_atl, ds_arc,
    lon_range=(lon_min, lon_max),
    lat_range=(lat_min, lat_max),
    month_range=(kd_month_start, kd_month_end),
)
del ds_atl, ds_arc


Filtered to months 4–10: Atlantic 1284 days, Arctic 1284 days
Atlantic valid: 932,830
Arctic valid:   1,397,434
Merged valid:   2,330,264


## 4. Save merged KD490 as GeoTIFF (WGS84)


In [20]:
kd_wgs84_path = str(out_dir / f"KD490_norge_{kd_start_year}-{kd_end_year}_wgs84.tif")

mnk.light.save_wgs84_geotiff(kd_merged, lons, lats, kd_wgs84_path)
del kd_merged


Saved WGS84 mosaic: KD490_norge_2020-2025_wgs84.tif


## 5. Reproject to EPSG:25833 at 1000 m and save (before fill)

In [21]:
kd_25833_path = str(out_dir / f"KD490_norge_{kd_start_year}-{kd_end_year}_25833.tif")
output_bounds = [dem_left, dem_bottom, dem_right, dem_top]

mnk.light.reproject_to_25833(kd_wgs84_path, kd_25833_path, output_bounds)


KD490 25833 — shape: (1537, 1222), valid: 647,570/1,878,214
Saved (unfilled): KD490_norge_2020-2025_25833.tif


PosixPath('KD490_norge_2020-2025_25833.tif')

## 6. Gap-fill KD490 using GDAL FillNodata

In [22]:
kd_filled_path = str(out_dir / f"KD490_norge_{kd_start_year}-{kd_end_year}_filled_25833.tif")

mnk.light.fill_kd490(kd_25833_path, kd_filled_path, output_bounds)


Fine fill (maxSearchDist=200)...
.100 - done.
0...10...20...30...40...50...60...70...80...90...Coarse fill (20x downsample, maxSearchDist=500)...
100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90..Filled — valid: 1,875,828/1,878,214
Saved: KD490_norge_2020-2025_filled_25833.tif


PosixPath('KD490_norge_2020-2025_filled_25833.tif')

## 7. Compute photic zone at DEM resolution (50 m)

Resample filled KD490 (1 km) to the DEM grid (50 m) using GDAL, then apply:

$$z_{photic} = \frac{\ln(100)}{K_{d490}} \approx \frac{4.605}{K_{d490}}$$

A pixel is **photic** (1) if `|depth| < z_photic`, **aphotic** (0) otherwise.
Processed block-by-block to avoid loading the full DEM into memory.

In [23]:
photic_path = str(out_dir / f"nisjedata-fotisk-sone-kd{kd_start_year}-{kd_end_year}_norge_2026_25833.tif")

In [24]:
mnk.light.compute_photic_zone(
    dem_path=dem_url,
    kd_path=kd_filled_path,
    out_path=photic_path,
    output_bounds=output_bounds,
    dem_xsize=dem_xsize,
    dem_ysize=dem_ysize,
)


KD490 resampled to DEM grid: 24431 x 30735
Photic  pixels: 7,438,034
Aphotic pixels: 30,148,600
Saved: nisjedata-fotisk-sone-kd2020-2025_norge_2026_25833.tif


PosixPath('nisjedata-fotisk-sone-kd2020-2025_norge_2026_25833.tif')

## 8. Vectorize photic zone - Create a pure vector version

Pad nodata by 1 pixel (so land subtraction clips cleanly at the coast),
vectorize with GDAL, subtract land, and save as GeoParquet + PostGIS.

In [25]:
# Pad the photic raster: dilate valid pixels by 1 into nodata
NODATA = -1

with rio.open(photic_path) as src:
    photic_arr = src.read(1)
    photic_profile = src.profile.copy()

photic_padded = mnk.vectorize.pad_nodata(photic_arr, nodata=NODATA)
print(f"Boundary pixels added: {(photic_padded != NODATA).sum() - (photic_arr != NODATA).sum():,}")
del photic_arr

# Write padded raster for vectorization
photic_padded_path = str(out_dir / f"fotisk-sone-kd{kd_start_year}-{kd_end_year}_padded.tif")
with rio.open(photic_padded_path, "w", **photic_profile) as dst:
    dst.write(photic_padded, 1)
del photic_padded, photic_profile
print(f"Saved padded raster: {photic_padded_path}")


Boundary pixels added: 1,138,606
Saved padded raster: fotisk-sone-kd2020-2025_padded.tif


In [26]:
# Vectorize padded photic raster
gdf_photic = mnk.vectorize.vectorize_raster_to_gdf(photic_padded_path, field_name="photic_int", epsg=25833, nodata=NODATA)

PHOTIC_NAMES = {1: "Eufotisk", 0: "Afotisk"}
gdf_photic["namn"] = gdf_photic["photic_int"].map(PHOTIC_NAMES)
print(f"Vectorized: {len(gdf_photic):,} polygons")
gdf_photic.head()


Vectorized: 37,478 polygons


,photic_int,geometry,namn
0,1,"POLYGON ((967500 7942300, 967500 7942150, 9676...",Eufotisk
1,1,"POLYGON ((970400 7942250, 970400 7942200, 9703...",Eufotisk
2,0,"POLYGON ((971650 7942200, 971650 7942100, 9717...",Afotisk
3,0,"POLYGON ((971450 7942150, 971450 7942050, 9715...",Afotisk
4,1,"POLYGON ((959050 7942100, 959050 7942000, 9591...",Eufotisk


In [27]:
# Subtract land
land = gpd.read_parquet(
    "gs://niva-geodata/MarintNaturKart/aux/Basisdata_Landareal.geo.parquet"
)
gdf_photic = mnk.vectorize.subtract_land(gdf_photic, land)
del land
print(f"After land subtraction: {len(gdf_photic):,} polygons")


Subtracting land from 18,413 of 37,478 polygons...
After land subtraction: 37,478 polygons


In [28]:
# Save as GeoParquet and upload to PostGIS
CRS = "EPSG:25833"
fname = mnk.utils.to_filename(f"nisjedata-fotisk-sone-kd{kd_start_year}-{kd_end_year}", "norge", "2026", CRS.split(":")[1])
gdf_photic.to_file(f"{fname}.gpkg", layer="fotisk", driver="GPKG")
print(f"Saved: {fname} ({len(gdf_photic):,} polygons)")

mnk.utils.to_postgis(gdf_photic, fname)


Saved: nisjedata-fotisk-sone-kd2020-2025_norge_2026_25833 (37,478 polygons)
Table nisjedata_fotisk_sone_kd2020_2025_norge_2026 uploaded to PostGIS.
